# Submission v5 — XGBoost + SMOTE + Multi-Window (1 + 3 + 6 min)

Extracts features at three timescales and concatenates them:
- **1-min**: acute EDA spikes, fast HR response
- **3-min**: sustained stress (main window used in v1–v4)
- **6-min**: slow temperature drift, cumulative arousal buildup

Final feature matrix = 3 × 107 = 321 features.

**Note:** XGBoost ≥ 2.0 requires `early_stopping_rounds` in the constructor, not `.fit()`.

In [1]:
%pip install xgboost scikit-learn pandas numpy scipy imbalanced-learn -q


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from scipy import stats as spstats
from sklearn.model_selection import StratifiedKFold, LeaveOneGroupOut
from sklearn.metrics import balanced_accuracy_score
from sklearn.impute import SimpleImputer
import xgboost as xgb
from imblearn.over_sampling import SMOTE
from collections import Counter

TRAIN_DATA  = pd.read_csv('train-sensor.csv')
TRAIN_LABEL = pd.read_csv('train-label.csv')
TEST_DATA   = pd.read_csv('test-sensor.csv')
TEST_LABEL  = pd.read_csv('test-label.csv')

TRAIN_LABEL['timestamp'] = TRAIN_LABEL['timestamp'].astype(float)
TEST_LABEL['timestamp']  = TEST_LABEL['timestamp'].astype(str).str.strip().astype(float)

print('TRAIN_DATA shape :', TRAIN_DATA.shape)
print('TRAIN_LABEL shape:', TRAIN_LABEL.shape)
print('TEST_DATA shape  :', TEST_DATA.shape)
print('TEST_LABEL shape :', TEST_LABEL.shape)
print()
print('Stress distribution:')
print(TRAIN_LABEL['stress'].value_counts().sort_index())


TRAIN_DATA shape : (4694400, 8)
TRAIN_LABEL shape: (815, 4)
TEST_DATA shape  : (5921280, 8)
TEST_LABEL shape : (1028, 4)

Stress distribution:
stress
0.0    162
1.0     66
2.0    587
Name: count, dtype: int64


In [3]:
SENSOR_COLS = ['accel_x','accel_y','accel_z','eda','heart_rate','temperature']

def compute_pid_stats(sensor_df):
    d = {}
    for pid, grp in sensor_df.groupby('pid'):
        d[pid] = {}
        for c in SENSOR_COLS:
            v  = grp[c].dropna().values.astype(float)
            d[pid][c+'_mu'] = float(np.mean(v)) if len(v)>0 else 0.0
            d[pid][c+'_sd'] = max(float(np.std(v)) if len(v)>0 else 1.0, 1e-6)
    return d

train_pid_stats = compute_pid_stats(TRAIN_DATA)
test_pid_stats  = compute_pid_stats(TEST_DATA)
global_mu = {c: np.median([train_pid_stats[p][c+'_mu'] for p in train_pid_stats]) for c in SENSOR_COLS}
global_sd = {c: np.median([train_pid_stats[p][c+'_sd'] for p in train_pid_stats]) for c in SENSOR_COLS}

def znorm(col_series, mu, sd):
    v = col_series.dropna().values.astype(float)
    return (v - mu) / sd if len(v) > 0 else np.array([])

def extract_window_features(label_df, sensor_df, pid_stats, window_ms, prefix):
    half_ms  = window_ms // 2
    third_ms = window_ms // 3
    sensor_by_pid = {pid: grp.sort_values('timestamp').reset_index(drop=True)
                     for pid, grp in sensor_df.groupby('pid')}
    rows = []
    for _, lrow in label_df.iterrows():
        pid = lrow['pid']; ts = float(lrow['timestamp']); lid = lrow['id']
        feat = {'id': lid}
        sg = sensor_by_pid.get(pid)
        if sg is None: rows.append(feat); continue
        ta = sg['timestamp'].values
        wa  = sg.loc[(ta>=ts-window_ms)&(ta<=ts),           SENSOR_COLS]
        wf  = sg.loc[(ta>=ts-window_ms)&(ta<ts-half_ms),    SENSOR_COLS]
        wl  = sg.loc[(ta>=ts-half_ms)  &(ta<=ts),           SENSOR_COLS]
        wt1 = sg.loc[(ta>=ts-window_ms)&(ta<ts-2*third_ms), SENSOR_COLS]
        wt3 = sg.loc[(ta>=ts-third_ms) &(ta<=ts),           SENSOR_COLS]
        for c in SENSOR_COLS:
            mu = pid_stats.get(pid,{}).get(c+'_mu', global_mu[c])
            sd = pid_stats.get(pid,{}).get(c+'_sd', global_sd[c])
            va=znorm(wa[c],mu,sd); vf=znorm(wf[c],mu,sd); vl=znorm(wl[c],mu,sd)
            vt1=znorm(wt1[c],mu,sd); vt3=znorm(wt3[c],mu,sd)
            p = f'{prefix}_{c}'
            if len(va)==0:
                for s in ['mean','std','min','max','median','skew','kurt','range',
                          'q25','q75','iqr','delta','slope','t1_mean','t3_mean','t3t1_delta','cv']:
                    feat[f'{p}_{s}'] = np.nan
                continue
            feat[f'{p}_mean']       = float(np.mean(va))
            feat[f'{p}_std']        = float(np.std(va))
            feat[f'{p}_min']        = float(np.min(va))
            feat[f'{p}_max']        = float(np.max(va))
            feat[f'{p}_median']     = float(np.median(va))
            feat[f'{p}_skew']       = float(spstats.skew(va))     if len(va)>2 else 0.0
            feat[f'{p}_kurt']       = float(spstats.kurtosis(va)) if len(va)>2 else 0.0
            feat[f'{p}_range']      = float(np.max(va)-np.min(va))
            feat[f'{p}_q25']        = float(np.percentile(va,25))
            feat[f'{p}_q75']        = float(np.percentile(va,75))
            feat[f'{p}_iqr']        = float(np.percentile(va,75)-np.percentile(va,25))
            feat[f'{p}_delta']      = float(np.mean(vl)-np.mean(vf)) if len(vf)>0 and len(vl)>0 else 0.0
            feat[f'{p}_slope']      = float(np.polyfit(np.linspace(0,1,len(va)),va,1)[0]) if len(va)>2 else 0.0
            feat[f'{p}_t1_mean']    = float(np.mean(vt1)) if len(vt1)>0 else 0.0
            feat[f'{p}_t3_mean']    = float(np.mean(vt3)) if len(vt3)>0 else 0.0
            feat[f'{p}_t3t1_delta'] = feat[f'{p}_t3_mean'] - feat[f'{p}_t1_mean']
            rv = wa[c].dropna().values.astype(float); rm = np.mean(rv)
            feat[f'{p}_cv']         = float(np.std(rv)/abs(rm)) if abs(rm)>1e-6 else 0.0
        ax=wa['accel_x'].values; ay=wa['accel_y'].values; az=wa['accel_z'].values
        if len(ax)>0:
            mag=np.sqrt(ax**2+ay**2+az**2)
            feat[f'{prefix}_accel_mag_mean']=float(np.mean(mag))
            feat[f'{prefix}_accel_mag_std'] =float(np.std(mag))
            feat[f'{prefix}_accel_mag_max'] =float(np.max(mag))
        else:
            feat[f'{prefix}_accel_mag_mean']=feat[f'{prefix}_accel_mag_std']=feat[f'{prefix}_accel_mag_max']=np.nan
        emu=pid_stats.get(pid,{}).get('eda_mu',global_mu['eda'])
        esd=pid_stats.get(pid,{}).get('eda_sd',global_sd['eda'])
        hmu=pid_stats.get(pid,{}).get('heart_rate_mu',global_mu['heart_rate'])
        hsd=pid_stats.get(pid,{}).get('heart_rate_sd',global_sd['heart_rate'])
        ez=znorm(wa['eda'],emu,esd); hz=znorm(wa['heart_rate'],hmu,hsd)
        if len(ez)>2 and len(hz)>2:
            n=min(len(ez),len(hz))
            feat[f'{prefix}_eda_hr_product']=float(np.mean(ez[:n]*hz[:n]))
            feat[f'{prefix}_eda_hr_corr']   =float(np.corrcoef(ez[:n],hz[:n])[0,1])
        else:
            feat[f'{prefix}_eda_hr_product']=feat[f'{prefix}_eda_hr_corr']=0.0
        rows.append(feat)
    return pd.DataFrame(rows).set_index('id')

WINDOWS = {'1min': 60_000, '3min': 180_000, '6min': 360_000}

all_train_feats = []
all_test_feats  = []
for wname, wms in WINDOWS.items():
    print(f'Extracting {wname} window...')
    tf = extract_window_features(TRAIN_LABEL, TRAIN_DATA, train_pid_stats, wms, wname)
    xf = extract_window_features(TEST_LABEL,  TEST_DATA,  test_pid_stats,  wms, wname)
    print(f'  train: {tf.shape}  test: {xf.shape}')
    all_train_feats.append(tf)
    all_test_feats.append(xf)

train_features = pd.concat(all_train_feats, axis=1)
test_features  = pd.concat(all_test_feats,  axis=1)
print(f'\nCombined train: {train_features.shape}')
print(f'Combined test : {test_features.shape}')


Extracting 1min window...
  train: (815, 107)  test: (1028, 107)
Extracting 3min window...
  train: (815, 107)  test: (1028, 107)
Extracting 6min window...
  train: (815, 107)  test: (1028, 107)

Combined train: (815, 321)
Combined test : (1028, 321)


In [4]:
tli    = TRAIN_LABEL.set_index('id')
y      = tli['stress'].astype(int)
groups = tli['pid']

imputer    = SimpleImputer(strategy='median')
X_imp      = pd.DataFrame(imputer.fit_transform(train_features),
                           columns=train_features.columns, index=train_features.index)
X_test_imp = pd.DataFrame(imputer.transform(test_features),
                           columns=test_features.columns, index=test_features.index)

counts        = Counter(y)
total         = len(y)
n_cls         = len(counts)
class_weights = {c: total/(n_cls*cnt) for c,cnt in counts.items()}
sample_weights = np.array([class_weights[yi] for yi in y])

print('X_imp shape:', X_imp.shape)
print('Class weights:', {k: round(v,3) for k,v in class_weights.items()})

def make_smote(X_tr, y_tr, seed=42):
    cnt1 = Counter(y_tr).get(1, 0)
    if cnt1 < 2: return X_tr, y_tr
    k = max(1, min(5, cnt1-1))
    return SMOTE(k_neighbors=k, random_state=seed).fit_resample(X_tr, y_tr)

def get_sw(y_arr):
    return np.array([class_weights[int(yi)] for yi in y_arr])


X_imp shape: (815, 321)
Class weights: {1: 4.116, 0: 1.677, 2: 0.463}


In [5]:
# XGBoost >= 2.0: early_stopping_rounds goes in the constructor, NOT in .fit()
XGB_PARAMS = dict(
    n_estimators          = 800,
    learning_rate         = 0.02,
    max_depth             = 5,
    subsample             = 0.7,
    colsample_bytree      = 0.7,
    reg_alpha             = 0.3,
    reg_lambda            = 1.0,
    min_child_weight      = 3,
    early_stopping_rounds = 50,
    objective             = 'multi:softprob',
    num_class             = 3,
    eval_metric           = 'mlogloss',
    n_jobs                = -1,
    verbosity             = 0,
)

print('=== LOPO CV — XGBoost + SMOTE + Multi-Window (1+3+6 min) ===')
logo = LeaveOneGroupOut()
lopo_scores = []

for tr_idx, val_idx in logo.split(X_imp, y, groups):
    pid_val = groups.iloc[val_idx[0]]
    y_val   = y.iloc[val_idx]
    if len(y_val.unique()) < 2:
        print(f'  Skip {pid_val}: only 1 class'); continue

    X_tr = X_imp.iloc[tr_idx].values
    y_tr = y.iloc[tr_idx].values
    X_va = X_imp.iloc[val_idx].values

    X_sm, y_sm = make_smote(X_tr, y_tr)
    sw_sm = get_sw(y_sm)

    m = xgb.XGBClassifier(**{**XGB_PARAMS, 'random_state': 42})
    m.fit(X_sm, y_sm,
          sample_weight = sw_sm,
          eval_set      = [(X_va, y_val)],
          verbose       = False)

    sc = balanced_accuracy_score(y_val, m.predict(X_va))
    print(f'  Leave out {pid_val}: {sc:.4f}  (n={len(val_idx)}, classes={sorted(y_val.unique())})')
    lopo_scores.append(sc)

print(f'\nXGBoost LOPO = {np.mean(lopo_scores):.4f} +/- {np.std(lopo_scores):.4f}')
print('v4 reference (3min only) = 0.4316  (target to beat)')


=== LOPO CV — XGBoost + SMOTE + Multi-Window (1+3+6 min) ===
  Leave out 43JW: 0.2582  (n=93, classes=[np.int64(0), np.int64(2)])
  Leave out C8Q6: 0.4401  (n=152, classes=[np.int64(0), np.int64(2)])
  Leave out DT5C: 0.3022  (n=90, classes=[np.int64(0), np.int64(1), np.int64(2)])
  Leave out F1ZM: 0.4104  (n=137, classes=[np.int64(1), np.int64(2)])
  Leave out HDS9: 0.6944  (n=135, classes=[np.int64(0), np.int64(2)])
  Leave out P4DZ: 0.3192  (n=144, classes=[np.int64(0), np.int64(1), np.int64(2)])
  Leave out TPQI: 0.4241  (n=64, classes=[np.int64(0), np.int64(2)])

XGBoost LOPO = 0.4070 +/- 0.1335
v4 reference (3min only) = 0.4316  (target to beat)


In [6]:
print('=== Final ensemble: XGBoost, 3 seeds x 5 folds ===')
SEEDS = [42, 7, 123]
all_test_proba = []
all_cv_scores  = []

for seed in SEEDS:
    skf        = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    seed_proba = np.zeros((len(X_test_imp), 3))
    fold_scores = []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_imp, y)):
        X_tr = X_imp.iloc[tr_idx].values
        y_tr = y.iloc[tr_idx].values
        X_va = X_imp.iloc[val_idx].values
        y_va = y.iloc[val_idx]

        X_sm, y_sm = make_smote(X_tr, y_tr, seed=seed)
        sw_sm = get_sw(y_sm)

        # early_stopping_rounds in constructor (XGBoost >= 2.0)
        m = xgb.XGBClassifier(**{**XGB_PARAMS,
                                  'random_state': seed,
                                  'early_stopping_rounds': 80})
        m.fit(X_sm, y_sm,
              sample_weight = sw_sm,
              eval_set      = [(X_va, y_va)],
              verbose       = False)

        sc = balanced_accuracy_score(y_va, m.predict(X_va))
        fold_scores.append(sc)
        seed_proba += m.predict_proba(X_test_imp.values)
        print(f'  Seed {seed} Fold {fold+1}: val BA = {sc:.4f}')

    seed_proba /= 5
    all_test_proba.append(seed_proba)
    all_cv_scores.append(np.mean(fold_scores))
    print(f'  Seed {seed} mean CV = {np.mean(fold_scores):.4f}')

print(f'\nEnsemble CV = {np.mean(all_cv_scores):.4f}')

final_proba = np.mean(all_test_proba, axis=0)
final_preds = np.argmax(final_proba, axis=1).astype(int)

print('\nPrediction distribution:')
for u, c in zip(*np.unique(final_preds, return_counts=True)):
    print(f'  class {u}: {c}')


=== Final ensemble: XGBoost, 3 seeds x 5 folds ===
  Seed 42 Fold 1: val BA = 0.7969
  Seed 42 Fold 2: val BA = 0.8034
  Seed 42 Fold 3: val BA = 0.8083
  Seed 42 Fold 4: val BA = 0.8247
  Seed 42 Fold 5: val BA = 0.8904
  Seed 42 mean CV = 0.8248
  Seed 7 Fold 1: val BA = 0.8189
  Seed 7 Fold 2: val BA = 0.8192
  Seed 7 Fold 3: val BA = 0.8656
  Seed 7 Fold 4: val BA = 0.8163
  Seed 7 Fold 5: val BA = 0.7917
  Seed 7 mean CV = 0.8223
  Seed 123 Fold 1: val BA = 0.7979
  Seed 123 Fold 2: val BA = 0.7252
  Seed 123 Fold 3: val BA = 0.8555
  Seed 123 Fold 4: val BA = 0.8371
  Seed 123 Fold 5: val BA = 0.8047
  Seed 123 mean CV = 0.8041

Ensemble CV = 0.8171

Prediction distribution:
  class 0: 293
  class 1: 54
  class 2: 681


In [7]:
submission = pd.DataFrame({'id': TEST_LABEL['id'].values, 'stress': final_preds})
submission.to_csv('submission_v5_multiwin.csv', index=False)
print('submission_v5_multiwin.csv saved!')
print(submission.head(10))


submission_v5_multiwin.csv saved!
     id  stress
0  1227       2
1  1228       2
2  1229       2
3  1230       2
4  1231       0
5  1232       0
6  1233       2
7  1234       2
8  1235       2
9  1236       1
